In [ ]:
import time
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from forecast_checks import ForecastValidator, MakeTrainingData

In [ ]:
#MAE is the main metric in the report
SCORING_METRIC = "neg_mean_absolute_error"

In [ ]:
#Walk forward, same folds as neural_network.ipynb: retrain at the start of every month and forecast that month.
#Helsingfors starts in mid-July, so September is the first month with Finnish training data
trainer = MakeTrainingData()
FIRST_TEST = "2022-09-01"
folds = trainer.walk_forward(FIRST_TEST)
for fold in folds:
    train_dates, test_dates = (trainer.df.loc[fold[part]["feature"].index, "date"] for part in ("train", "test"))
    print(f"fit: {train_dates.min().date()} - {train_dates.max().date()}, test: {test_dates.min().date()} - {test_dates.max().date()}")

In [ ]:
#Hyperparameter tuning with GridSearchCV, once, inside the first fold's training data so no test month is used.
#Same three 2-week validation windows as neural_network.ipynb; each is trained on everything up to 6 days before it

NUM_TREES=[100,500,1000]
LEARNING_RATE=[0.025, 0.05, 0.075]
MAX_DEPTH=[5,10,15]

BASE_PARAMS = dict(enable_categorical=True, tree_method="hist", random_state=42)
param_grid = {"n_estimators": NUM_TREES, "learning_rate": LEARNING_RATE, "max_depth": MAX_DEPTH}

VAL_DAYS, N_WINDOWS = 14, 3
X_tune, y_tune = folds[0]["train"]["feature"], folds[0]["train"]["target"]
tune_end = trainer.df.loc[X_tune.index, "date"].max()
val_windows = trainer.walk_forward(tune_end - pd.Timedelta(days=VAL_DAYS * N_WINDOWS - 1), freq=f"{VAL_DAYS}D", until=tune_end)
#GridSearchCV takes the windows as positional row numbers in X_tune
cv = [(X_tune.index.get_indexer(w["train"]["feature"].index), X_tune.index.get_indexer(w["test"]["feature"].index))
      for w in val_windows]
search = GridSearchCV(xgb.XGBRegressor(**BASE_PARAMS), param_grid, cv=cv,
                      scoring=SCORING_METRIC, refit=False,
                      verbose=True)
search.fit(X_tune, y_tune)
print("Best params:", search.best_params_)

In [ ]:
#TRAIN: walk forward with the tuned parameters, refit on everything known at the start of each month and forecast that month

preds = []
for fold in folds:
    fold_start = time.time()
    X_fit, X_pred = fold["train"]["feature"], fold["test"]["feature"]
    model = xgb.XGBRegressor(**BASE_PARAMS, **search.best_params_)
    model.fit(X_fit, fold["train"]["target"])
    preds.append(pd.Series(model.predict(X_pred), index=X_pred.index))
    print(f"{trainer.df.loc[X_pred.index, 'date'].min():%Y-%m}: {len(X_fit)} training rows, {time.time() - fold_start:.0f} s")
pred = pd.concat(preds)
X_test = pd.concat([fold["test"]["feature"] for fold in folds])
y_test = pd.concat([fold["test"]["target"] for fold in folds])

#Score every model on the same hours as neural_network.ipynb: drop test rows where a consumption lag is missing
scored = X_test.filter(like="consumption").notna().all(axis=1)
X_test, y_test, pred = X_test[scored], y_test[scored], pred[scored]

In [ ]:
#TEST

forecastValidator = (ForecastValidator(y_test, groups=X_test["location"])
                     .add("Same hour last week", X_test["consumption_7days_ago"], baseline=True)
                     .add("XGBoost", pred))
forecastValidator.summary()

In [ ]:
forecastValidator.save_to_json("results/xgboost.json")

In [ ]:
forecastValidator.per_group()

In [ ]:
#Error per test month, to see how it changes from autumn through winter to spring
forecastValidator.per_period(trainer.df.loc[y_test.index, "date"])

In [ ]:
forecastValidator.plot_predicted_vs_actual_consumption(no_baseline=True)

In [ ]:
forecastValidator.plot_consumption_over_time(trainer.df.loc[y_test.index, "time"], no_baseline=True)